In [1]:
# Pin versions to avoid transformers / huggingface_hub compatibility issues
!pip install -q \
    "transformers==4.47.1" \
    "huggingface_hub==0.26.5" \
    "tokenizers>=0.20" \
    sentence-transformers \
    chromadb \
    accelerate \
    bitsandbytes \
    pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.8/447.8 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.

In [2]:
# ============================================================
# GLOBAL CONFIGURATION — edit here, nowhere else
# ============================================================

# --- Paths ---
DB_PATH        = "warehouse.db"
CHROMA_PATH    = "./chroma_wms"
COLLECTION_NAME = "wms_schema"

# --- Models ---
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_NAME    = "Qwen/Qwen2.5-3B-Instruct"

# --- Synthetic data volumes ---
DATA_CFG = {
    "seed":                       42,     # random seed for reproducibility
    "clear_existing":             True,   # wipe tables before inserting
    "n_locations":                120,
    "n_items":                    250,
    "n_dealers":                  80,
    "n_users":                    30,
    "n_cases":                    300,
    "case_lines_min":             1,
    "case_lines_max":             6,
    "n_orders":                   600,
    "order_lines_min":            1,
    "order_lines_max":            8,
    "prob_case_received":         0.75,
    "prob_reconcile_system":      0.80,
    "prob_order_cancelled":       0.03,
    "inventory_locations_sample": 100,
    "inventory_items_sample":     200,
}

# --- Text-to-SQL ---
SCHEMA_RETRIEVE_K  = 4    # number of schema chunks to retrieve per query
MAX_NEW_TOKENS     = 160  # max tokens for SQL generation

print("✅ Config loaded.")
print(f"   DB: {DB_PATH}  |  LLM: {LLM_NAME}  |  Embed: {EMBED_MODEL}")

✅ Config loaded.
   DB: warehouse.db  |  LLM: Qwen/Qwen2.5-3B-Instruct  |  Embed: sentence-transformers/all-MiniLM-L6-v2


In [3]:
import sqlite3
from pathlib import Path

# Full DDL — stored as a module-level constant so schema_sql is available
# to later cells (embedding, RAG prompt building, etc.)
SCHEMA_SQL = r"""
CREATE TABLE location_file (
  location_id   TEXT PRIMARY KEY,
  zone          TEXT,
  aisle         TEXT,
  bay           TEXT,
  level         TEXT,
  position      TEXT,
  location_type TEXT,   -- PICK / RESERVE / DOCK / QA / STAGE
  is_active     INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE item_file (
  item_id       TEXT PRIMARY KEY,
  item_desc     TEXT NOT NULL,
  uom           TEXT NOT NULL,  -- EA / CS etc.
  case_pack_qty INTEGER,
  each_per_case INTEGER,
  is_active     INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE dealer_file (
  dealer_id   TEXT PRIMARY KEY,
  dealer_name TEXT NOT NULL,
  address1    TEXT,
  city        TEXT,
  state       TEXT,
  zip         TEXT,
  phone       TEXT,
  is_active   INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE warehouse_user_file (
  user_id    TEXT PRIMARY KEY,
  username   TEXT UNIQUE NOT NULL,
  full_name  TEXT,
  role       TEXT NOT NULL,  -- ADMIN / SUPERVISOR / OPERATOR
  is_active  INTEGER NOT NULL DEFAULT 1,
  created_ts TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE inventory_file (
  inventory_id    INTEGER PRIMARY KEY AUTOINCREMENT,
  location_id     TEXT NOT NULL,
  item_id         TEXT NOT NULL,
  on_hand_qty     DECIMAL(18,3) NOT NULL DEFAULT 0,
  allocated_qty   DECIMAL(18,3) NOT NULL DEFAULT 0,
  available_qty   DECIMAL(18,3) NOT NULL DEFAULT 0,
  last_updated_ts TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (location_id) REFERENCES location_file(location_id),
  FOREIGN KEY (item_id)     REFERENCES item_file(item_id),
  CONSTRAINT ck_inventory_qty
    CHECK (on_hand_qty >= 0 AND allocated_qty >= 0 AND available_qty >= 0)
);

CREATE UNIQUE INDEX ux_inventory_loc_item ON inventory_file(location_id, item_id);

CREATE TABLE order_header (
  order_id   TEXT PRIMARY KEY,
  dealer_id  TEXT NOT NULL,
  order_date DATE NOT NULL,
  status     TEXT NOT NULL
               CHECK (status IN ('OPEN','ALLOCATED','PICKED','SHIPPED','CANCELLED')),
  priority   INTEGER DEFAULT 5,
  created_by TEXT,
  created_ts TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (dealer_id)  REFERENCES dealer_file(dealer_id),
  FOREIGN KEY (created_by) REFERENCES warehouse_user_file(user_id)
);

CREATE TABLE order_detail (
  order_line_id INTEGER PRIMARY KEY AUTOINCREMENT,
  order_id      TEXT NOT NULL,
  item_id       TEXT NOT NULL,
  ordered_qty   DECIMAL(18,3) NOT NULL,
  allocated_qty DECIMAL(18,3) NOT NULL DEFAULT 0,
  picked_qty    DECIMAL(18,3) NOT NULL DEFAULT 0,
  shipped_qty   DECIMAL(18,3) NOT NULL DEFAULT 0,
  FOREIGN KEY (order_id) REFERENCES order_header(order_id),
  FOREIGN KEY (item_id)  REFERENCES item_file(item_id),
  CONSTRAINT ck_order_qty
    CHECK (ordered_qty >= 0 AND allocated_qty >= 0
           AND picked_qty >= 0 AND shipped_qty >= 0)
);

CREATE INDEX ix_order_detail_order ON order_detail(order_id);
CREATE INDEX ix_order_detail_item  ON order_detail(item_id);

CREATE TABLE work_assignment_header (
  wa_id        TEXT PRIMARY KEY,
  wa_type      TEXT NOT NULL
                 CHECK (wa_type IN ('PICK','PUTAWAY','REPLEN','COUNT')),
  status       TEXT NOT NULL
                 CHECK (status IN ('NEW','ASSIGNED','IN_PROGRESS','COMPLETED','CLOSED')),
  created_by   TEXT NOT NULL,
  assigned_to  TEXT,
  created_ts   TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  started_ts   TIMESTAMP,
  completed_ts TIMESTAMP,
  FOREIGN KEY (created_by)  REFERENCES warehouse_user_file(user_id),
  FOREIGN KEY (assigned_to) REFERENCES warehouse_user_file(user_id)
);

CREATE TABLE work_assignment_detail (
  wa_line_id           INTEGER PRIMARY KEY AUTOINCREMENT,
  wa_id                TEXT NOT NULL,
  task_type            TEXT NOT NULL
                         CHECK (task_type IN ('PICK','PUTAWAY','MOVE','COUNT')),
  item_id              TEXT NOT NULL,
  from_location_id     TEXT,
  to_location_id       TEXT,
  requested_qty        DECIMAL(18,3) NOT NULL,
  confirmed_qty        DECIMAL(18,3),
  source_order_id      TEXT,
  source_order_line_id INTEGER,
  status               TEXT NOT NULL
                         CHECK (status IN ('OPEN','CONFIRMED','CANCELLED')),
  confirmed_by         TEXT,
  confirmed_ts         TIMESTAMP,
  FOREIGN KEY (wa_id)            REFERENCES work_assignment_header(wa_id),
  FOREIGN KEY (item_id)          REFERENCES item_file(item_id),
  FOREIGN KEY (from_location_id) REFERENCES location_file(location_id),
  FOREIGN KEY (to_location_id)   REFERENCES location_file(location_id),
  FOREIGN KEY (confirmed_by)     REFERENCES warehouse_user_file(user_id),
  FOREIGN KEY (source_order_id)  REFERENCES order_header(order_id)
);

CREATE INDEX ix_wa_detail_wa ON work_assignment_detail(wa_id);

CREATE TABLE case_header (
  case_id                 TEXT PRIMARY KEY,
  container_id            TEXT,
  status                  TEXT NOT NULL
                            CHECK (status IN ('EXPECTED','RECEIVED','RECONCILED','CANCELLED')),
  expected_date           DATE NOT NULL,
  case_received_date      DATE,
  reconciliation_due_date DATE NOT NULL,
  reconciliation_date     DATE,
  reconciled_by           TEXT,  -- user_id OR 'SYSTEM'
  created_ts              TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  updated_ts              TIMESTAMP,
  CONSTRAINT ck_reconciled_by_valid
    CHECK (reconciled_by IS NULL OR reconciled_by = 'SYSTEM' OR LENGTH(reconciled_by) > 0)
);

CREATE TABLE case_detail (
  case_line_id   INTEGER PRIMARY KEY AUTOINCREMENT,
  case_id        TEXT NOT NULL,
  item_id        TEXT NOT NULL,
  qty_to_receive DECIMAL(18,3) NOT NULL,
  received_qty   DECIMAL(18,3) NOT NULL DEFAULT 0,
  FOREIGN KEY (case_id) REFERENCES case_header(case_id),
  FOREIGN KEY (item_id) REFERENCES item_file(item_id),
  CONSTRAINT ck_case_qty_nonnegative
    CHECK (qty_to_receive >= 0 AND received_qty >= 0),
  CONSTRAINT ck_case_received_not_exceed
    CHECK (received_qty <= qty_to_receive)
);

CREATE INDEX ix_case_detail_case ON case_detail(case_id);
CREATE INDEX ix_case_detail_item ON case_detail(item_id);

CREATE TABLE invoice_header (
  invoice_id   TEXT PRIMARY KEY,
  order_id     TEXT NOT NULL,
  dealer_id    TEXT NOT NULL,
  invoice_date DATE NOT NULL,
  status       TEXT NOT NULL
                 CHECK (status IN ('OPEN','POSTED','CANCELLED')),
  total_amount DECIMAL(18,2),
  created_by   TEXT,
  created_ts   TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  FOREIGN KEY (order_id)   REFERENCES order_header(order_id),
  FOREIGN KEY (dealer_id)  REFERENCES dealer_file(dealer_id),
  FOREIGN KEY (created_by) REFERENCES warehouse_user_file(user_id)
);

CREATE TABLE invoice_detail (
  invoice_line_id INTEGER PRIMARY KEY AUTOINCREMENT,
  invoice_id      TEXT NOT NULL,
  item_id         TEXT NOT NULL,
  qty             DECIMAL(18,3) NOT NULL,
  unit_price      DECIMAL(18,2) NOT NULL,
  ext_amount      DECIMAL(18,2) NOT NULL,
  FOREIGN KEY (invoice_id) REFERENCES invoice_header(invoice_id),
  FOREIGN KEY (item_id)    REFERENCES item_file(item_id)
);

CREATE INDEX ix_invoice_detail_invoice ON invoice_detail(invoice_id);
"""


def create_schema(db_path: str, schema_sql: str) -> None:
    """Drop existing DB file and recreate schema from scratch."""
    # Remove old DB so we always start clean
    db_file = Path(db_path)
    if db_file.exists():
        db_file.unlink()

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.executescript(schema_sql)
    conn.commit()

    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
    ).fetchall()
    conn.close()

    print(f"✅ Schema created at: {db_file.resolve()}")
    print("   Tables:", ", ".join(t[0] for t in tables))


create_schema(DB_PATH, SCHEMA_SQL)

✅ Schema created at: /content/warehouse.db
   Tables: case_detail, case_header, dealer_file, inventory_file, invoice_detail, invoice_header, item_file, location_file, order_detail, order_header, sqlite_sequence, warehouse_user_file, work_assignment_detail, work_assignment_header


In [4]:
import random
import sqlite3
from datetime import date, datetime, timedelta

random.seed(DATA_CFG["seed"])


# ── Utility helpers ──────────────────────────────────────────

def d_to_str(d: date | None) -> str | None:
    return d.strftime("%Y-%m-%d") if d else None


def dt_to_str(dt: datetime | None) -> str | None:
    return dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None


def rand_datetime(base_date: date, hour_lo: int = 6, hour_hi: int = 20) -> datetime:
    """Random datetime on base_date between hour_lo and hour_hi."""
    return datetime(
        base_date.year, base_date.month, base_date.day,
        random.randint(hour_lo, hour_hi),
        random.randint(0, 59),
        random.randint(0, 59),
    )


def make_id(prefix: str, i: int, width: int = 4) -> str:
    return f"{prefix}-{i:0{width}d}"


def sample(seq: list, k: int) -> list:
    return random.sample(seq, k=min(k, len(seq)))


# ── Generators ───────────────────────────────────────────────

def _gen_locations(cfg: dict) -> list[tuple]:
    rows = []
    for i in range(1, cfg["n_locations"] + 1):
        if i <= 5:
            loc_id = f"DOCK-{i:02d}"
            rows.append((loc_id, "DOCK", None, None, None, None, "DOCK", 1))
        elif i <= 15:
            loc_id = f"STAGE-{i - 5:02d}"
            rows.append((loc_id, "STAGE", None, None, None, None, "STAGE", 1))
        else:
            zone     = random.choice(["PICK", "RES"])
            aisle    = f"{random.choice(list('ABCDEFGH'))}{random.randint(1, 20):02d}"
            bay      = f"{random.randint(1, 30):02d}"
            level    = f"{random.randint(1, 5):02d}"
            pos      = f"{random.randint(1, 6):02d}"
            loc_type = "PICK" if zone == "PICK" else "RESERVE"
            loc_id   = f"{loc_type}-{aisle}-{bay}-{level}-{pos}"
            rows.append((loc_id, zone, aisle, bay, level, pos, loc_type, 1))
    return rows


def _gen_items(cfg: dict) -> list[tuple]:
    nouns = ["Filter", "Brake Pad", "Wiper Blade", "Spark Plug",
             "Belt", "Hose", "Bearing", "Sensor", "Rotor", "Battery"]
    sizes = ["Small", "Medium", "Large", "XL", "18in", "20in", "24in", "Front", "Rear"]
    rows  = []
    for i in range(1, cfg["n_items"] + 1):
        pack = random.choice([6, 8, 10, 12, 16, 24, 36])
        rows.append((
            f"SKU-{1000 + i}",
            f"{random.choice(nouns)} - {random.choice(sizes)}",
            "EA", pack, pack, 1,
        ))
    return rows


def _gen_dealers(cfg: dict) -> list[tuple]:
    cities = [
        ("Chicago", "IL", "60601"), ("Evanston", "IL", "60201"),
        ("Aurora", "IL", "60505"),  ("Naperville", "IL", "60540"),
        ("Joliet", "IL", "60431"),  ("Elgin", "IL", "60120"),
    ]
    streets = ["Main", "Lake", "Oak", "Pine", "Market"]
    rows = []
    for i in range(1, cfg["n_dealers"] + 1):
        city, st, zp = random.choice(cities)
        rows.append((
            make_id("D", i, 3),
            f"Dealer {i:03d}",
            f"{random.randint(10, 9999)} {random.choice(streets)} St",
            city, st, zp,
            f"{random.randint(200,999)}-555-{random.randint(1000,9999)}",
            1,
        ))
    return rows


def _gen_users(cfg: dict) -> list[tuple]:
    roles = ["OPERATOR", "SUPERVISOR", "ADMIN"]
    rows  = []
    for i in range(1, cfg["n_users"] + 1):
        role = random.choices(roles, weights=[0.75, 0.20, 0.05], k=1)[0]
        rows.append((make_id("U", i, 3), f"user{i:03d}", f"User {i:03d}", role, 1))
    return rows


def _gen_inventory(location_ids: list, item_ids: list, cfg: dict) -> list[tuple]:
    rows = []
    for loc in sample(location_ids, cfg["inventory_locations_sample"]):
        for item in sample(item_ids, cfg["inventory_items_sample"]):
            on_hand   = random.randint(0, 400)
            allocated = random.randint(0, min(30, on_hand)) if on_hand > 0 else 0
            rows.append((loc, item, float(on_hand), float(allocated), float(on_hand - allocated)))
    return rows


def _gen_cases(
    item_ids: list, user_ids: list, today: date, cfg: dict
) -> tuple[list, list]:
    headers, details = [], []
    for i in range(1, cfg["n_cases"] + 1):
        case_id       = make_id("CASE", i, 4)
        container_id  = f"CONT-{random.randint(1, max(1, cfg['n_cases'] // 10))}"
        expected_date = today - timedelta(days=random.randint(1, 140))
        created_ts    = rand_datetime(expected_date - timedelta(days=5))

        received           = random.random() < cfg["prob_case_received"]
        case_received_date = expected_date + timedelta(days=random.randint(0, 3)) if received else None
        due_date           = (case_received_date + timedelta(days=7)) if case_received_date \
                             else (expected_date + timedelta(days=21))

        reconciled = random.random() < (0.9 if due_date <= today else 0.1)
        if reconciled:
            max_date      = min(today, due_date + timedelta(days=7))
            rec_date      = due_date + timedelta(days=random.randint(0, max(0, (max_date - due_date).days)))
            reconciled_by = "SYSTEM" if random.random() < cfg["prob_reconcile_system"] \
                            else random.choice(user_ids)
            status        = "RECONCILED"
            updated_ts    = rand_datetime(rec_date)
        else:
            rec_date      = None
            reconciled_by = None
            status        = "RECEIVED" if case_received_date else "EXPECTED"
            updated_ts    = None

        headers.append((
            case_id, container_id, status,
            d_to_str(expected_date), d_to_str(case_received_date),
            d_to_str(due_date),      d_to_str(rec_date),
            reconciled_by,           dt_to_str(created_ts), dt_to_str(updated_ts),
        ))

        for item_id in sample(item_ids, random.randint(cfg["case_lines_min"], cfg["case_lines_max"])):
            qty_to_receive = round(random.uniform(5, 120), 3)
            if case_received_date:
                recv_qty = qty_to_receive if random.random() < 0.85 \
                           else round(qty_to_receive * random.uniform(0.5, 0.99), 3)
            else:
                recv_qty = round(qty_to_receive * random.uniform(0.0, 0.25), 3)
            details.append((case_id, item_id, qty_to_receive, min(recv_qty, qty_to_receive)))

    return headers, details


def _gen_orders(
    dealer_ids: list, user_ids: list, item_ids: list, today: date, cfg: dict
) -> tuple[list, list]:
    statuses  = ["OPEN", "ALLOCATED", "PICKED", "SHIPPED"]
    headers, details = [], []
    for i in range(1, cfg["n_orders"] + 1):
        order_id   = make_id("ORD", i, 5)
        order_date = today - timedelta(days=random.randint(1, 90))
        status     = "CANCELLED" if random.random() < cfg["prob_order_cancelled"] \
                     else random.choices(statuses, weights=[0.35, 0.25, 0.20, 0.20], k=1)[0]
        headers.append((
            order_id, random.choice(dealer_ids),
            d_to_str(order_date), status,
            random.randint(1, 10), random.choice(user_ids),
        ))
        for item_id in sample(item_ids, random.randint(cfg["order_lines_min"], cfg["order_lines_max"])):
            details.append((order_id, item_id, round(random.uniform(1, 20), 3)))
    return headers, details


def _gen_work_assignments(
    orders: list, order_to_lines: dict, location_ids: list,
    inv_locs: list, user_ids: list, today: date
) -> tuple[list, list]:
    wa_statuses = ["NEW", "ASSIGNED", "IN_PROGRESS", "COMPLETED", "CLOSED"]
    stage_locs  = [l for l in location_ids if l.startswith("STAGE-")]
    headers, details = [], []

    for counter, (order_id, order_status) in enumerate(orders, start=1):
        if order_status == "CANCELLED":
            continue
        wa_id     = make_id("WA", counter, 5)
        wa_status = random.choices(wa_statuses, weights=[0.05, 0.20, 0.30, 0.30, 0.15], k=1)[0]
        assigned  = random.choice(user_ids)
        headers.append((wa_id, "PICK", wa_status, random.choice(user_ids), assigned))

        for line_id, item_id, qty in order_to_lines.get(order_id, []):
            confirmed    = wa_status in ("COMPLETED", "CLOSED") and random.random() < 0.9
            det_status   = "CONFIRMED" if confirmed else "OPEN"
            confirmed_ts = dt_to_str(rand_datetime(today - timedelta(days=random.randint(0, 30)))) \
                           if confirmed else None
            details.append((
                wa_id, "PICK", item_id,
                random.choice(inv_locs or location_ids),
                random.choice(stage_locs) if stage_locs else None,
                float(qty),
                float(qty) if confirmed else None,
                order_id, int(line_id),
                det_status,
                assigned if confirmed else None,
                confirmed_ts,
            ))
    return headers, details


def _gen_invoices(
    orders: list, order_to_lines: dict, user_ids: list, today: date
) -> tuple[list, list]:
    headers, details = [], []
    counter = 1
    for order_id, dealer_id, status in orders:
        if status != "SHIPPED" or random.random() > 0.95:
            continue
        invoice_id = make_id("INV", counter, 5)
        counter   += 1
        total      = 0.0
        for _, item_id, qty in order_to_lines.get(order_id, []):
            price = round(random.uniform(5, 120), 2)
            ext   = round(float(qty) * price, 2)
            total += ext
            details.append((invoice_id, item_id, float(qty), price, ext))
        inv_status = random.choices(["OPEN", "POSTED"], weights=[0.25, 0.75], k=1)[0]
        headers.append((
            invoice_id, order_id, dealer_id,
            d_to_str(today - timedelta(days=random.randint(0, 20))),
            inv_status, round(total, 2), random.choice(user_ids),
        ))
    return headers, details


# ── Main seeder ───────────────────────────────────────────────

def seed_database(db_path: str, cfg: dict) -> None:
    """Insert all synthetic data into the WMS database."""
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur  = conn.cursor()
    today = datetime.now().date()

    # 1. Master tables
    cur.executemany(
        "INSERT INTO location_file "
        "(location_id,zone,aisle,bay,level,position,location_type,is_active) "
        "VALUES (?,?,?,?,?,?,?,?)",
        _gen_locations(cfg),
    )
    cur.executemany(
        "INSERT INTO item_file "
        "(item_id,item_desc,uom,case_pack_qty,each_per_case,is_active) "
        "VALUES (?,?,?,?,?,?)",
        _gen_items(cfg),
    )
    cur.executemany(
        "INSERT INTO dealer_file "
        "(dealer_id,dealer_name,address1,city,state,zip,phone,is_active) "
        "VALUES (?,?,?,?,?,?,?,?)",
        _gen_dealers(cfg),
    )
    cur.executemany(
        "INSERT INTO warehouse_user_file "
        "(user_id,username,full_name,role,is_active) VALUES (?,?,?,?,?)",
        _gen_users(cfg),
    )
    conn.commit()

    # Load IDs into memory for FK references
    loc_ids  = [r[0] for r in cur.execute("SELECT location_id FROM location_file WHERE is_active=1")]
    item_ids = [r[0] for r in cur.execute("SELECT item_id FROM item_file WHERE is_active=1")]
    dlr_ids  = [r[0] for r in cur.execute("SELECT dealer_id FROM dealer_file WHERE is_active=1")]
    usr_ids  = [r[0] for r in cur.execute("SELECT user_id FROM warehouse_user_file WHERE is_active=1")]

    # 2. Inventory
    inv_locs = sample(loc_ids, cfg["inventory_locations_sample"])
    cur.executemany(
        "INSERT OR IGNORE INTO inventory_file "
        "(location_id,item_id,on_hand_qty,allocated_qty,available_qty) VALUES (?,?,?,?,?)",
        _gen_inventory(inv_locs, item_ids, cfg),
    )
    conn.commit()

    # 3. Cases
    case_hdrs, case_dtls = _gen_cases(item_ids, usr_ids, today, cfg)
    cur.executemany(
        "INSERT INTO case_header "
        "(case_id,container_id,status,expected_date,case_received_date,"
        "reconciliation_due_date,reconciliation_date,reconciled_by,created_ts,updated_ts) "
        "VALUES (?,?,?,?,?,?,?,?,?,?)",
        case_hdrs,
    )
    cur.executemany(
        "INSERT INTO case_detail (case_id,item_id,qty_to_receive,received_qty) VALUES (?,?,?,?)",
        case_dtls,
    )
    conn.commit()

    # 4. Orders
    ord_hdrs, ord_dtls = _gen_orders(dlr_ids, usr_ids, item_ids, today, cfg)
    cur.executemany(
        "INSERT INTO order_header "
        "(order_id,dealer_id,order_date,status,priority,created_by) VALUES (?,?,?,?,?,?)",
        ord_hdrs,
    )
    cur.executemany(
        "INSERT INTO order_detail (order_id,item_id,ordered_qty) VALUES (?,?,?)",
        ord_dtls,
    )
    conn.commit()

    # Build order→lines lookup for WA and invoice generation
    order_to_lines: dict = {}
    for line_id, oid, iid, qty in cur.execute(
        "SELECT order_line_id,order_id,item_id,ordered_qty FROM order_detail"
    ):
        order_to_lines.setdefault(oid, []).append((line_id, iid, qty))

    # 5. Work assignments
    orders = list(cur.execute("SELECT order_id,status FROM order_header"))
    wa_hdrs, wa_dtls = _gen_work_assignments(
        orders, order_to_lines, loc_ids, inv_locs, usr_ids, today
    )
    cur.executemany(
        "INSERT INTO work_assignment_header "
        "(wa_id,wa_type,status,created_by,assigned_to) VALUES (?,?,?,?,?)",
        wa_hdrs,
    )
    cur.executemany(
        "INSERT INTO work_assignment_detail "
        "(wa_id,task_type,item_id,from_location_id,to_location_id,"
        "requested_qty,confirmed_qty,source_order_id,source_order_line_id,"
        "status,confirmed_by,confirmed_ts) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
        wa_dtls,
    )
    conn.commit()

    # 6. Invoices
    all_orders = list(cur.execute("SELECT order_id,dealer_id,status FROM order_header"))
    inv_hdrs, inv_dtls = _gen_invoices(all_orders, order_to_lines, usr_ids, today)
    cur.executemany(
        "INSERT INTO invoice_header "
        "(invoice_id,order_id,dealer_id,invoice_date,status,total_amount,created_by) "
        "VALUES (?,?,?,?,?,?,?)",
        inv_hdrs,
    )
    cur.executemany(
        "INSERT INTO invoice_detail "
        "(invoice_id,item_id,qty,unit_price,ext_amount) VALUES (?,?,?,?,?)",
        inv_dtls,
    )
    conn.commit()

    # Summary
    print("✅ Synthetic data inserted. Row counts:")
    for tbl in [
        "location_file", "item_file", "dealer_file", "warehouse_user_file",
        "inventory_file",
        "case_header", "case_detail",
        "order_header", "order_detail",
        "work_assignment_header", "work_assignment_detail",
        "invoice_header", "invoice_detail",
    ]:
        n = cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
        print(f"   {tbl:<30s} {n:>6,}")

    conn.close()


seed_database(DB_PATH, DATA_CFG)

✅ Synthetic data inserted. Row counts:
   location_file                     120
   item_file                         250
   dealer_file                        80
   warehouse_user_file                30
   inventory_file                 20,000
   case_header                       300
   case_detail                     1,061
   order_header                      600
   order_detail                    2,746
   work_assignment_header            588
   work_assignment_detail          2,682
   invoice_header                    102
   invoice_detail                    465


In [5]:
import sqlite3
import pandas as pd


def preview_table(db_path: str, query: str, label: str = "") -> pd.DataFrame:
    """Execute a SELECT query and return a DataFrame."""
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    df = pd.read_sql_query(query, conn)
    conn.close()
    if label:
        print(f"── {label} ({len(df)} rows) ──")
    return df


preview_table(
    DB_PATH,
    "SELECT * FROM case_header ORDER BY created_ts DESC LIMIT 10;",
    label="case_header (latest 10)",
)

── case_header (latest 10) (10 rows) ──


,case_id,container_id,status,expected_date,case_received_date,reconciliation_due_date,reconciliation_date,reconciled_by,created_ts,updated_ts
0,CASE-0108,CONT-5,RECONCILED,2026-04-05,None,2026-04-26,2026-04-26,U-013,2026-03-31 10:54:35,2026-04-26 18:43:21
1,CASE-0168,CONT-24,RECEIVED,2026-04-05,2026-04-06,2026-04-13,None,None,2026-03-31 07:04:11,None
2,CASE-0175,CONT-2,EXPECTED,2026-04-05,None,2026-04-26,None,None,2026-03-31 06:09:25,None
3,CASE-0104,CONT-12,RECEIVED,2026-04-04,2026-04-06,2026-04-13,None,None,2026-03-30 19:50:01,None
4,CASE-0190,CONT-27,RECEIVED,2026-04-04,2026-04-06,2026-04-13,None,None,2026-03-30 19:37:06,None
5,CASE-0065,CONT-11,RECEIVED,2026-04-04,2026-04-04,2026-04-11,None,None,2026-03-30 16:28:03,None
6,CASE-0035,CONT-18,EXPECTED,2026-04-04,None,2026-04-25,None,None,2026-03-30 15:34:58,None
7,CASE-0279,CONT-25,RECEIVED,2026-04-04,2026-04-07,2026-04-14,None,None,2026-03-30 10:44:27,None
8,CASE-0192,CONT-1,RECEIVED,2026-04-03,2026-04-03,2026-04-10,None,None,2026-03-29 11:37:34,None
9,CASE-0151,CONT-8,EXPECTED,2026-04-03,None,2026-04-24,None,None,2026-03-29 09:09:47,None


In [6]:
import re
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer


# ── Schema parsing ────────────────────────────────────────────

def extract_table_chunks(schema_sql: str) -> list[dict]:
    """
    Parse DDL and return a list of dicts, one per table:
      { 'table': <name>, 'text': '<human-readable chunk>' }
    The text format is: 'Table: X. Columns: c1, c2. Definition: ...'
    """
    pattern = re.compile(
        r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?"
        r"([A-Za-z_][A-Za-z0-9_]*)\s*\((.*?)\)\s*;",
        re.IGNORECASE | re.DOTALL,
    )
    chunks = []
    for table, body in pattern.findall(schema_sql):
        body_clean = re.sub(r"\s+", " ", body).strip()
        col_names  = [
            re.sub(r"\s+", " ", p).strip().split()[0].strip('"`[]')
            for p in body.split(",")
            if p.strip()
            and not re.match(
                r"^\s*(CONSTRAINT|FOREIGN KEY|PRIMARY KEY|CHECK|UNIQUE)\b",
                p, flags=re.I,
            )
            and re.match(r"^\s*[A-Za-z_][A-Za-z0-9_]*", p)
        ]
        chunks.append({
            "table": table,
            "text":  f"Table: {table}. Columns: {', '.join(col_names)}. "
                     f"Definition: {body_clean}",
        })
    return chunks


# ── Embedding + ChromaDB storage ──────────────────────────────

def build_vector_store(
    schema_sql: str,
    embed_model_name: str,
    chroma_path: str,
    collection_name: str,
) -> tuple[chromadb.Collection, SentenceTransformer, list[dict]]:
    """
    Parse schema → embed → upsert into ChromaDB.
    Returns (collection, embedder, table_chunks) for later reuse.
    """
    table_chunks = extract_table_chunks(schema_sql)
    texts        = [c["text"] for c in table_chunks]
    print(f"   Parsed {len(table_chunks)} table chunks from schema.")

    embedder   = SentenceTransformer(embed_model_name)
    embeddings = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    embeddings = np.array(embeddings, dtype="float32")
    print(f"   Embeddings shape: {embeddings.shape}")

    client     = chromadb.PersistentClient(path=chroma_path)
    # Delete stale collection if it exists so we always have fresh embeddings
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    collection = client.create_collection(collection_name)
    collection.add(
        ids       = [f"table::{c['table']}" for c in table_chunks],
        documents = texts,
        embeddings= embeddings.tolist(),
        metadatas = [{"table": c["table"], "type": "ddl_table_chunk"} for c in table_chunks],
    )
    print(f"✅ ChromaDB ready — {collection.count()} vectors stored.")
    return collection, embedder, table_chunks


COLLECTION, EMBEDDER, TABLE_CHUNKS = build_vector_store(
    SCHEMA_SQL, EMBED_MODEL, CHROMA_PATH, COLLECTION_NAME
)

   Parsed 13 table chunks from schema.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   Embeddings shape: (13, 384)
✅ ChromaDB ready — 13 vectors stored.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def load_llm(
    model_name: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Load a causal LLM in 4-bit quantization.
    Falls back to float16 (no quantization) if bitsandbytes is unavailable.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    try:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=bnb_cfg,
            torch_dtype=torch.float16,
        )
        print(f"✅ LLM loaded with 4-bit quantization: {model_name}")
    except Exception as e:
        print(f"⚠️  4-bit loading failed ({e}). Falling back to float16.")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        print(f"✅ LLM loaded in float16: {model_name}")

    print(f"   CUDA available: {torch.cuda.is_available()}")
    return model, tokenizer


LLM, TOKENIZER = load_llm(LLM_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM loaded with 4-bit quantization: Qwen/Qwen2.5-3B-Instruct
   CUDA available: True


In [8]:
import re
import sqlite3
import pandas as pd
import torch

# Keywords that must never appear in generated SQL
_BANNED_KEYWORDS = [
    "DROP", "DELETE", "UPDATE", "INSERT", "ALTER", "TRUNCATE",
    "ATTACH", "DETACH", "PRAGMA", "REINDEX", "VACUUM", "CREATE",
]


# ── Step 1: Retrieve relevant schema chunks ───────────────────

def retrieve_schema(
    question: str,
    embedder: SentenceTransformer,
    collection: chromadb.Collection,
    k: int = SCHEMA_RETRIEVE_K,
) -> str:
    """Embed the question and return top-k schema chunks as a formatted string."""
    q_emb = embedder.encode([question], normalize_embeddings=True).tolist()
    res   = collection.query(query_embeddings=q_emb, n_results=k)
    lines = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        table = meta.get("table", "unknown") if isinstance(meta, dict) else "unknown"
        lines.append(f"- {table}: {doc}")
    return "\n".join(lines)


# ── Step 2: Build prompt ──────────────────────────────────────

def build_prompt(question: str, schema_context: str) -> str:
    """Construct the system + user prompt for SQL generation."""
    return (
        "You are a SQL expert.\n"
        "You must write a valid SQLite SELECT query only.\n"
        "Return ONLY the SQL query (no markdown, no explanation).\n\n"
        f"Schema context (most relevant tables):\n{schema_context}\n\n"
        f"User question:\n{question}\n\n"
        "Rules:\n"
        "- Use only tables/columns from the schema context above.\n"
        "- Prefer order_header/order_detail for order questions.\n"
        "- Filter by specific IDs when mentioned in the question.\n"
        "- Return only ONE SQL statement.\n"
    )


# ── Step 3: Generate SQL ──────────────────────────────────────

def _extract_sql(text: str) -> str:
    """Extract the last SELECT/WITH block from raw LLM output."""
    matches = list(re.finditer(r"\b(SELECT|WITH)\b", text, flags=re.IGNORECASE))
    if not matches:
        return text.strip()
    candidate = text[matches[-1].start():].split("\n\n")[0].strip()
    return candidate.replace("```sql", "").replace("```", "").strip()


def generate_sql(
    prompt: str,
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> str:
    """Run the LLM and return the extracted SQL string."""
    try:
        messages   = [
            {"role": "system", "content": "You are a helpful assistant that outputs only SQL."},
            {"role": "user",   "content": prompt},
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        input_text = prompt

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return _extract_sql(raw)


# ── Step 4: Validate SQL ──────────────────────────────────────

def validate_sql(sql: str) -> tuple[bool, str]:
    """Return (is_valid, reason). Rejects non-SELECT and dangerous keywords."""
    if not sql or not isinstance(sql, str):
        return False, "Empty SQL."
    s = sql.strip()
    if not re.match(r"^(SELECT|WITH)\b", s, flags=re.IGNORECASE):
        return False, "SQL must start with SELECT or WITH."
    if ";" in s.rstrip(";"):
        return False, "Multiple statements detected."
    upper = s.upper()
    for kw in _BANNED_KEYWORDS:
        if re.search(rf"\b{kw}\b", upper):
            return False, f"Forbidden keyword: '{kw}'."
    return True, "OK"


# ── Step 5: Execute SQL ───────────────────────────────────────

def run_sql(sql: str, db_path: str = DB_PATH) -> pd.DataFrame:
    """Execute a SELECT query and return results as a DataFrame."""
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    try:
        df = pd.read_sql_query(sql, conn)
    finally:
        conn.close()
    return df


# ── Step 6: Full pipeline with one auto-retry ─────────────────

def text_to_sql(
    question: str,
    embedder: SentenceTransformer     = EMBEDDER,
    collection: chromadb.Collection   = COLLECTION,
    model: AutoModelForCausalLM       = LLM,
    tokenizer: AutoTokenizer          = TOKENIZER,
    k: int                            = SCHEMA_RETRIEVE_K,
    retry: bool                       = True,
) -> dict:
    """
    Full RAG Text-to-SQL pipeline.

    Returns a dict with keys:
      question, sql, status ('OK' | 'OK_AFTER_RETRY' | 'REJECTED' | 'ERROR'), reason, df
    """
    schema_ctx = retrieve_schema(question, embedder, collection, k)
    prompt     = build_prompt(question, schema_ctx)
    sql        = generate_sql(prompt, model, tokenizer)

    ok, msg = validate_sql(sql)
    if not ok:
        return {"question": question, "sql": sql, "status": "REJECTED", "reason": msg, "df": None}

    try:
        df = run_sql(sql)
        return {"question": question, "sql": sql, "status": "OK", "reason": "Executed", "df": df}
    except Exception as e:
        if not retry:
            return {"question": question, "sql": sql, "status": "ERROR", "reason": str(e), "df": None}

        # One repair attempt: feed the error back to the LLM
        repair_prompt = (
            prompt
            + f"\n\nThe SQL above caused this SQLite error:\n{e}\n"
            + "Fix the SQL. Return ONLY the corrected SQL query."
        )
        sql2    = generate_sql(repair_prompt, model, tokenizer)
        ok2, msg2 = validate_sql(sql2)
        if not ok2:
            return {"question": question, "sql": sql2, "status": "REJECTED", "reason": msg2, "df": None}
        try:
            df2 = run_sql(sql2)
            return {"question": question, "sql": sql2, "status": "OK_AFTER_RETRY",
                    "reason": "Fixed and executed", "df": df2}
        except Exception as e2:
            return {"question": question, "sql": sql2, "status": "ERROR", "reason": str(e2), "df": None}


print("✅ Text-to-SQL pipeline ready. Call text_to_sql(<your question>).")

✅ Text-to-SQL pipeline ready. Call text_to_sql(<your question>).


In [9]:
QUESTIONS = [
    "How is CASE-0001 reconciled?",
]

for q in QUESTIONS:
    print("=" * 60)
    print(f"Q: {q}")
    result = text_to_sql(q)
    print(f"Status : {result['status']}")
    print(f"Reason : {result['reason']}")
    print(f"SQL    :\n{result['sql']}")
    if result["df"] is not None:
        display(result["df"].head(10))
    print()

Q: How is CASE-0001 reconciled?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Status : OK
Reason : Executed
SQL    :
SELECT th.case_id, th.container_id, th.status, th.expected_date, th.case_received_date, th.reconciliation_due_date, th.reconciliation_date, th.reconciled_by, th.updated_ts FROM case_header AS th WHERE th.case_id = 'CASE-0001'


,case_id,container_id,status,expected_date,case_received_date,reconciliation_due_date,reconciliation_date,reconciled_by,updated_ts
0,CASE-0001,CONT-13,RECONCILED,2025-12-13,2025-12-14,2025-12-21,2025-12-21,SYSTEM,2025-12-21 17:04:42


In [10]:
QUESTIONS = [
    "Which items have the most inventory on hand?",
]

for q in QUESTIONS:
    print("=" * 60)
    print(f"Q: {q}")
    result = text_to_sql(q)
    print(f"Status : {result['status']}")
    print(f"Reason : {result['reason']}")
    print(f"SQL    :\n{result['sql']}")
    if result["df"] is not None:
        display(result["df"].head(10))
    print()

Q: Which items have the most inventory on hand?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Status : OK
Reason : Executed
SQL    :
SELECT item_id FROM inventory_file GROUP BY item_id ORDER BY SUM(on_hand_qty) DESC LIMIT 1


,item_id
0,SKU-1059


In [11]:
QUESTIONS = [
    "Delete reconciled cases from case_header",
]

for q in QUESTIONS:
    print("=" * 60)
    print(f"Q: {q}")
    result = text_to_sql(q)
    print(f"Status : {result['status']}")
    print(f"Reason : {result['reason']}")
    print(f"SQL    :\n{result['sql']}")
    if result["df"] is not None:
        display(result["df"].head(10))
    print()

Q: Delete reconciled cases from case_header


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Status : REJECTED
Reason : Forbidden keyword: 'DELETE'.
SQL    :
SELECT query only.
Return ONLY the SQL query (no markdown, no explanation).': near ".": syntax error
Fix the SQL. Return ONLY the corrected SQL query.
assistant
DELETE FROM case_header WHERE status = 'RECONCILED'

